In [2]:
import pandas as pd
import os
import databento as db
from dotenv import load_dotenv

load_dotenv('./.env')

# trades from Rust decoded parquet schmea
trades_df = pd.read_parquet('../rust-ingest/pcap_samples/ny4-small-10m_trades.parquet')
trades_df["size"] = trades_df["size"].astype("Int64")

# comparison symbols between databento and Rust decoded parquet
symbol = ["SPY   230822P00436000"]
start = "2023-08-22T14:30:00.000000Z"
end = "2023-08-22T14:31:00.000000Z"

# trades filtered from Rust decoded parquet
compare_df = trades_df[trades_df['osi_symbol'].isin(symbol) & \
                      (trades_df['block_timestamp_utc'] >= start) & \
                        (trades_df['block_timestamp_utc'] <= end) & \
                        (trades_df['category'] == 'a') # actual trade messages
                        ]
print(compare_df.shape)

# databento trades schema filtered
client = db.Historical(os.getenv("DATABENTO_API_KEY"))  # uses DATABENTO_API_KEY env var
db_trades = client.timeseries.get_range(
    dataset="OPRA.PILLAR",
    schema="trades",    
    stype_in="raw_symbol",
    symbols=symbol,
    start=start,
    end=end,
    limit=1000
).to_df()
print(db_trades.shape)

(55, 20)
(55, 13)


In [ ]:
# compare_df trades head and tail
display(compare_df.head(2))
display(compare_df.tail(2))

# databento trades head and tail
display(db_trades.head(2))
display(db_trades.tail(2))

,packet_index,block_sequence,block_timestamp_ns,block_timestamp_utc,message_index_in_block,participant,category,type_code,indicator,symbol_root,osi_symbol,bid,ask,bid_size,ask_size,price,size,side,action,flags
81071,34692,355538012,1692714600170725888,2023-08-22T14:30:00.170725888Z,0,E,a,a,,SPY,SPY 230822P00436000,NaN,NaN,NaN,NaN,0.13,1.0,None,a,0.0
350772,150404,355622374,1692714600972599296,2023-08-22T14:30:00.972599296Z,0,B,a,I,,SPY,SPY 230822P00436000,NaN,NaN,NaN,NaN,0.13,10.0,None,I,0.0


,packet_index,block_sequence,block_timestamp_ns,block_timestamp_utc,message_index_in_block,participant,category,type_code,indicator,symbol_root,osi_symbol,bid,ask,bid_size,ask_size,price,size,side,action,flags
15223307,7002914,360517449,1692714658069873152,2023-08-22T14:30:58.069873152Z,6,B,a,I,,SPY,SPY 230822P00436000,NaN,NaN,NaN,NaN,0.15,2.0,None,I,0.0
15481044,7117978,360601423,1692714658869331200,2023-08-22T14:30:58.869331200Z,0,B,a,I,,SPY,SPY 230822P00436000,NaN,NaN,NaN,NaN,0.15,1.0,None,I,0.0


,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,flags,ts_in_delta,sequence,symbol
ts_recv,,,,,,,,,,,,,
2023-08-22 14:30:00.170935673+00:00,2023-08-22 14:30:00.170725888+00:00,0,24,654312762,T,N,0,0.13,1,194,0,355538012,SPY 230822P00436000
2023-08-22 14:30:00.972809829+00:00,2023-08-22 14:30:00.972599296+00:00,0,21,654312762,T,N,0,0.13,10,194,0,355622374,SPY 230822P00436000


,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,flags,ts_in_delta,sequence,symbol
ts_recv,,,,,,,,,,,,,
2023-08-22 14:30:58.070080852+00:00,2023-08-22 14:30:58.069873152+00:00,0,21,654312762,T,N,0,0.15,2,194,0,360517455,SPY 230822P00436000
2023-08-22 14:30:58.869538349+00:00,2023-08-22 14:30:58.869331200+00:00,0,21,654312762,T,N,0,0.15,1,194,0,360601423,SPY 230822P00436000


In [5]:
# trades from Rust decoded parquet schmea
trades_df = pd.read_parquet('../rust-ingest/pcap_samples/ny4-small-10m_trades.parquet')
print(trades_df['category'].value_counts(normalize=True))
trades_df["size"] = trades_df["size"].astype("Int64")

# comparison symbols between databento and Rust decoded parquet
symbol = ["SPY   230822P00436000"]
start = "2023-08-22T14:30:00.000000Z"
end = "2023-08-22T14:31:00.000000Z"

# trades filtered from Rust decoded parquet
compare_df = trades_df[trades_df['osi_symbol'].isin(symbol) & \
                      (trades_df['block_timestamp_utc'] >= start) & \
                        (trades_df['block_timestamp_utc'] <= end) & \
                        (trades_df['category'] != 'a') # actual trade messages
                        ]
print(compare_df.shape)
compare_df.head()

category
q    0.999720
a    0.000268
k    0.000012
Name: proportion, dtype: float64
(10818, 20)


,packet_index,block_sequence,block_timestamp_ns,block_timestamp_utc,message_index_in_block,participant,category,type_code,indicator,symbol_root,osi_symbol,bid,ask,bid_size,ask_size,price,size,side,action,flags
230,125,355512401,1692714600000281856,2023-08-22T14:30:00.000281856Z,0,E,q,A,A,SPY,SPY 230822P00436000,0.13,0.14,96.0,222.0,NaN,<NA>,None,None,NaN
1415,661,355512895,1692714600003264000,2023-08-22T14:30:00.003264000Z,0,Z,q,A,A,SPY,SPY 230822P00436000,0.13,0.14,1771.0,869.0,NaN,<NA>,None,None,NaN
3741,1618,355513774,1692714600006264832,2023-08-22T14:30:00.006264832Z,1,E,q,A,A,SPY,SPY 230822P00436000,0.13,0.14,17.0,222.0,NaN,<NA>,None,None,NaN
4803,2084,355514115,1692714600007444224,2023-08-22T14:30:00.007444224Z,2,I,q,,A,SPY,SPY 230822P00436000,0.13,0.14,1026.0,136.0,NaN,<NA>,None,None,NaN
4853,2103,355514137,1692714600007477248,2023-08-22T14:30:00.007477248Z,0,C,q,A,A,SPY,SPY 230822P00436000,0.13,0.14,152.0,100.0,NaN,<NA>,None,None,NaN
